# 🏦 Financial Market Research Agent with Web Search 📈

Welcome to our **Financial Market Research Agent with Web Search** tutorial! In this notebook, we'll demonstrate how to:

1. **Initialize** a project using Microsoft Foundry.
2. **Create an Agent** with the **WebSearchTool** for real-time web search.
3. **Ask real-world questions** about market trends, interest rates, and financial news.
4. **Retrieve and display** answers with real-time market data and inline citations.

## What Changed: Bing Grounding → Web Search Tool

The `BingGroundingAgentTool` has been replaced by the simpler `WebSearchTool`. Key differences:
- **No connection setup required** — `WebSearchTool` works out of the box (uses Bing under the hood).
- **Simpler API** — no `BingGroundingSearchToolParameters` or `BingGroundingSearchConfiguration`.
- **Optional location** — pass `WebSearchApproximateLocation` for geo-relevant results.
- **Inline citations** — responses include `url_citation` annotations automatically.

## Prerequisites
- An active Azure AI Foundry project with web search enabled at the subscription level.
- A `.env` file in the parent directory containing:
  ```bash
  AI_FOUNDRY_PROJECT_ENDPOINT=<your-ai-foundry-project-endpoint>
  AZURE_AI_MODEL_DEPLOYMENT_NAME=<supported-model>
  ```

## Let's Explore Real-Time Financial Research!
We'll integrate **Web Search** into our agent so it can gather current market data, interest rate news, and economic trends. This is perfect for financial advisors who need up-to-date information! 🎉

<br/>

## 🔐 Authentication Setup

Before running the next cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login --use-device-code
```

This will provide you with a device code and URL to authenticate in your browser, which is useful for:
- Remote development environments
- Systems without a default browser
- Corporate environments with strict security policies

After successful authentication, you can proceed with the notebook cells below.

## 1. Initial Setup
We'll load environment variables from `.env` and initialize our **AIProjectClient** to manage agents.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    WebSearchTool,
    WebSearchApproximateLocation,
)

# Load environment variables from parent .env
notebook_path = Path().absolute()
parent_dir = notebook_path.parent
load_dotenv(parent_dir.parent / '.env')

# Get configuration
project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

print(f"🔗 Project Endpoint: {project_endpoint[:50]}..." if project_endpoint else "❌ AI_FOUNDRY_PROJECT_ENDPOINT not set")
print(f"🤖 Model: {model_name}")

# Initialize AIProjectClient with AzureCliCredential
credential = AzureCliCredential()
project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=credential,
)
print("✅ Successfully initialized AIProjectClient")

🔗 Project Endpoint: https://kd-foundry.services.ai.azure.com/api/proje...
🤖 Model: gpt-4o
✅ Successfully initialized AIProjectClient


## 2. Create Web Search Financial Research Agent 🌐

We'll create an agent with the `WebSearchTool` — no Bing connection setup required. The tool enables the agent to search the public web and return grounded responses with inline citations.

In [2]:
def create_web_search_agent():
    """Create a financial research agent with WebSearchTool for real-time market data."""
    try:
        # Create WebSearchTool with optional location for geo-relevant results
        web_search_tool = WebSearchTool(
            user_location=WebSearchApproximateLocation(
                country="US",
                city="New York",
                region="New York",
            )
        )

        # Create financial research agent with web search
        agent = project_client.agents.create_version(
            agent_name="financial-market-research-agent",
            definition=PromptAgentDefinition(
                model=model_name,
                instructions="""
                    You are a financial market research assistant with web search capabilities.
                    Always:
                    1. Provide disclaimers that you are not a licensed financial advisor and this is not investment advice.
                    2. Encourage professional consultation for investment decisions.
                    3. Use web search to find current interest rates, market news, and economic trends.
                    4. Provide brief, accurate summaries with source citations.
                    5. Include relevant sources and publication dates for market data.
                    6. Never recommend specific investments or predict market movements.
                """,
                tools=[web_search_tool],
            ),
            description="Financial market research agent with web search",
        )

        print(f"🎉 Created financial research agent with Web Search")
        print(f"   Name: {agent.name}, Version: {agent.version}")
        return agent

    except Exception as e:
        print(f"❌ Error creating web search agent: {e}")
        import traceback
        traceback.print_exc()
        return None

# Create our web search financial research agent
web_search_agent = create_web_search_agent()

🎉 Created financial research agent with Web Search
   Name: financial-market-research-agent, Version: 1


## 3. Researching Financial Markets 💬

We'll use the Responses API to query the agent. The agent will search the web and return grounded answers with URL citations.

In [3]:
def ask_financial_research_question(agent, user_query):
    """Ask a financial research question using the Foundry Responses API with streaming."""
    try:
        openai_client = project_client.get_openai_client()

        print(f"📨 Researching: '{user_query}'\n")

        # Use the Responses API with streaming for real-time output
        stream_response = openai_client.responses.create(
            stream=True,
            tool_choice="required",
            input=user_query,
            extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        )

        full_text = ""
        citations = []

        for event in stream_response:
            if event.type == "response.output_text.delta":
                full_text += event.delta
            elif event.type == "response.output_item.done":
                if event.item.type == "message":
                    item = event.item
                    if item.content[-1].type == "output_text":
                        text_content = item.content[-1]
                        for annotation in text_content.annotations:
                            if annotation.type == "url_citation":
                                citations.append(annotation.url)

        print(f"🤖 {full_text[:500]}...")
        if citations:
            print(f"\n📎 Sources:")
            for url in citations[:5]:
                print(f"   • {url}")
        print("-" * 60 + "\n")
        return full_text

    except Exception as e:
        print(f"❌ Error with financial research: {e}")
        import traceback
        traceback.print_exc()
        return None


if web_search_agent:
    # Financial research questions for FSI customers
    questions = [
        "What are the current Federal Reserve interest rate decisions and their impact on mortgage rates?",
        "What are the latest trends in banking sector performance and major bank stock movements?",
        "What are financial experts saying about inflation forecasts for the upcoming year?",
    ]

    for q in questions:
        ask_financial_research_question(web_search_agent, q)

📨 Researching: 'What are the current Federal Reserve interest rate decisions and their impact on mortgage rates?'



🤖 I am not a licensed financial advisor and this is not investment advice. Please consult a qualified financial or mortgage professional for individual guidance.

Below is a current overview of the Federal Reserve’s interest rate decisions as of mid‑May 2026 and how they are influencing U.S. mortgage rates:

---

###  Federal Reserve Interest Rate Decision (Mid‑May 2026)

- At its **April 28–29, 2026** Federal Open Market Committee (FOMC) meeting, the Federal Reserve **held the federal funds targe...

📎 Sources:
   • https://www.usbank.com/investing/financial-perspectives/market-news/federal-reserve-tapering-asset-purchases.html
   • https://www.usbank.com/investing/financial-perspectives/market-news/federal-reserve-tapering-asset-purchases.html
   • https://www.cnbc.com/2026/04/29/fed-interest-rate-decision-april-2026.html
   • https://www.sofrrate.com/policy-rates
   • https://www.federalreserve.gov/releases/h15/
------------------------------------------------------------



📨 Researching: 'What are the latest trends in banking sector performance and major bank stock movements?'



🤖 Below is a detailed, structured briefing (in markdown format) on the **latest trends in banking sector performance** and **notable bank stock movements** as of mid‑May 2026.

---

**Disclaimer**:  
I am not a licensed financial advisor, and this is not investment advice. Consult a qualified financial professional before making any investment decisions.

---

##  Section 1: Banking Sector Performance Trends

###  1. Resilient Fundamentals Amid Macroeconomic Challenges  
- **Stable asset quality a...

📎 Sources:
   • https://www.moodys.com/web/en/us/insights/credit-risk/outlooks/banking-2026.html
   • https://www.spglobal.com/market-intelligence/en/news-insights/research/2026/01/key-themes-2026-banking-risk
   • https://www.accenture.com/us-en/insights/banking/accenture-banking-trends-2026
   • https://www.accenture.com/us-en/insights/banking/accenture-banking-trends-2026
   • https://www.bankingdive.com/news/banking-trends-outlook-2026/810544/
-----------------------------------------

📨 Researching: 'What are financial experts saying about inflation forecasts for the upcoming year?'



🤖 Here is what financial experts and institutions are currently saying—based on the latest available data—about inflation forecasts for the upcoming year (2026). Please note the following important disclaimers before proceeding:

**Disclaimer**: I am not a licensed financial advisor, and this is not investment advice. For decisions related to investing or personal finance, please consult a qualified professional.

---

##  Summary of Inflation Forecasts for 2026

### 1. Federal Reserve (FOMC) Proj...

📎 Sources:
   • https://finance.yahoo.com/economy/policy/articles/fed-quietly-altered-march-inflation-100000735.html
   • https://www.financialexpress.com/market/global-markets-fomc-meeting-2026-fed-raises-inflation-forecast-holds-rates-5-key-dot-plot-takeaways-4177359/
   • https://fredblog.stlouisfed.org/2026/03/fomc-summary-of-economic-projections-march-2026/
   • https://www.cnbc.com/2026/04/01/oecd-predicts-higher-inflation-than-fedwhat-that-means-for-your-money.html
   • https://www

## 4. Cleanup & Best Practices

### Best Practices
1. **Accuracy** – Web search results may include partial info. Encourage verification with credible sources.
2. **Citations** – The agent automatically includes `url_citation` annotations. Display them for transparency.
3. **Location** – Use `WebSearchApproximateLocation` to get geo-relevant financial news (e.g., US market hours, local regulation).
4. **Privacy** – Avoid sending sensitive data in prompts that might be forwarded to search.
5. **Admin Control** – Web search can be disabled at the subscription level via Azure CLI.
6. **Evaluations** – Use `azure-ai-evaluation` for iterative improvement.

In [4]:
# def cleanup_agent(agent):
#     """Clean up agent using Foundry API."""
#     if agent:
#         try:
#             project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
#             print(f"🗑️ Deleted agent: {agent.name} (Version: {agent.version})")
#         except Exception as e:
#             print(f"❌ Error cleaning up agent: {e}")
#     else:
#         print("No agent to clean up.")

# # Uncomment to delete the agent
# cleanup_agent(web_search_agent)

## Summary

In this notebook, we demonstrated:

| Concept | Details |
|---------|---------|
| **WebSearchTool** | Replaces `BingGroundingAgentTool` — simpler API, no connection setup |
| **DefaultAzureCredential** | Production-ready auth (works with `az login`, managed identity, etc.) |
| **Responses API** | Streaming with `openai.responses.create(stream=True)` and `agent_reference` |
| **Citations** | URL citations via `url_citation` annotations in response output items |
| **Location** | `WebSearchApproximateLocation` for geo-relevant financial results |

### Migration from BingGroundingAgentTool

```python
# OLD (deprecated)
from azure.ai.projects.models import BingGroundingAgentTool, BingGroundingSearchToolParameters, BingGroundingSearchConfiguration
bing_tool = BingGroundingAgentTool(
    bing_grounding=BingGroundingSearchToolParameters(
        search_configurations=[BingGroundingSearchConfiguration(project_connection_id=conn_id)]
    )
)

# NEW (recommended)
from azure.ai.projects.models import WebSearchTool, WebSearchApproximateLocation
web_tool = WebSearchTool(user_location=WebSearchApproximateLocation(country="US", city="New York"))
```

### Next Steps
- **Notebook 5**: [AI Search with Vector/Hybrid Search](./5-agents-aisearch.ipynb)
- **Notebook 7**: [MCP Tools](./7-mcp-tools.ipynb)